In [1]:
# !pip install -q transformers datasets peft accelerate pandas

In [2]:
import pandas as pd
import json
import re
import os
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

In [3]:
INPUT_FILENAME = "ddq_document_v2.xlsx" 
OUTPUT_FILENAME = "train_data.jsonl"

def clean_text(text):
    if not isinstance(text, str): return ""
    return text.strip().replace('\n', ' ')

def convert_to_training_data(file_path, output_path):
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    training_pairs = []

    print(f"[*] Đang xử lý file: {file_path}")
    
    for _, row in df.iterrows():
        description = clean_text(row.get('description', ''))
        
        examples_raw = row.get('examples', '')
        
        if not description or not isinstance(examples_raw, str):
            continue
            
        example_list = re.split(r'[,\n]+', examples_raw)
        
        for ex in example_list:
            ex = clean_text(ex)
            if len(ex) > 5: 
                entry = {
                    "input": f"paraphrase: {ex}", 
                    "target": description
                }
                training_pairs.append(entry)

    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in training_pairs:
            json.dump(entry, f, ensure_ascii=False)
            f.write('\n')
            
    print(f"Đã tạo thành công {len(training_pairs)} cặp dữ liệu training!")
    print(f"File lưu tại: {output_path}")

current_dir = os.getcwd()
file_path = os.path.join(current_dir, INPUT_FILENAME)

if os.path.exists(file_path):
    convert_to_training_data(file_path, OUTPUT_FILENAME)
else:
    xlsx_files = [f for f in os.listdir(current_dir) if f.endswith('.xlsx')]
    if xlsx_files:
        print(f"⚠ Không thấy {INPUT_FILENAME}, đang dùng file: {xlsx_files[0]}")
        convert_to_training_data(os.path.join(current_dir, xlsx_files[0]), OUTPUT_FILENAME)
    else:
        print(f"X LỖI: Không tìm thấy file Excel nào")

[*] Đang xử lý file: e:\proton-intern\javisAI\6804_ddq\Text2SQL_Template\models\train\ddq_document_v2.xlsx
Đã tạo thành công 1604 cặp dữ liệu training!
File lưu tại: train_data.jsonl


In [4]:
# Load Dataset & Tokenizer
dataset = load_dataset("json", data_files="train_data.jsonl")

full_dataset = dataset['train'].train_test_split(test_size=0.1)

print("Mẫu dữ liệu:", full_dataset['train'][0])

model_id = "chieunq/vietnamese-sentence-paraphase"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = full_dataset.map(preprocess_function, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Mẫu dữ liệu: {'input': 'paraphrase: Xem lịch sử giao dịch mua sắm phát sinh phí giao dịch', 'target': 'Người dùng muốn tra cứu lịch sử giao dịch mua sắm theo phí giao dịch.'}


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/1443 [00:00<?, ? examples/s]

Map:   0%|          | 0/161 [00:00<?, ? examples/s]

In [ ]:
# LoRA config & train
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir="./fine_tuned_banking_ai",
    learning_rate=1e-3,
    per_device_train_batch_size=8, 
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True, 
    use_cpu=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Bắt đầu training...")
trainer.train()

trainable params: 1,769,472 || all params: 255,442,176 || trainable%: 0.6927


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21028\3868293769.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Bắt đầu training...


Epoch,Training Loss,Validation Loss


In [ ]:
def generate_answer(text):
    input_text = f"paraphrase: {text}"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs["input_ids"], max_length=128, num_beams=5)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 50)
print("Input: Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi")
print("Output:", generate_answer("Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi"))
print("-" * 50)

model.save_pretrained("my_banking_lora_adapter")
tokenizer.save_pretrained("my_banking_lora_adapter")
# !zip -r my_banking_ai.zip my_banking_lora_adapter

--------------------------------------------------
Input: Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi
Output: Người dùng muốn tra cứu lịch sử giao dịch chuyển tiền theo khoảng thời gian xác định.
--------------------------------------------------


('my_banking_lora_adapter\\tokenizer_config.json',
 'my_banking_lora_adapter\\special_tokens_map.json',
 'my_banking_lora_adapter\\spiece.model',
 'my_banking_lora_adapter\\added_tokens.json',
 'my_banking_lora_adapter\\tokenizer.json')